<a href="https://colab.research.google.com/github/Davidson-Open-University/csc203-labs/blob/main/notebooks/CSC203-Lab6-Matrices-with-NumPy.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 203 Lab 6: Matrices with NumPy

**Study Session 6: Matrices and Determinants** · Davidson Open University Global

**In this lab you will:**
- create matrices with NumPy and find their order (shape);
- add, multiply and transpose matrices, and see that AB ≠ BA;
- compute determinants and inverses;
- solve a system of linear equations with matrices.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the examples and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short Python function and test it.
- **Part D – Apply and reflect** (about 10 minutes): use your tools on your own example and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `csc203-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

*This lab is practice to help you learn and prepare for the assignment. It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), 6) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k) + '=' + _norm(v) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), 6) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('csc203|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False)
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '6.1': {'h': '512a570f3ee077d9', 'hint': 'Use P.shape: (number of rows, number of columns).'},
    '6.2': {'h': 'e6db44f226f726ed', 'hint': 'Use @. P is 2 × 3 and Q is 3 × 2, so PQ is 2 × 2.', 'ok': 'Check one entry by hand: row 1 of P times column 1 of Q.'},
    '6.3': {'h': '257dfea8d0a43e47', 'hint': 'The transpose swaps rows and columns: P.T.'},
    '6.4': {'h': '9e8d0c35b508de2e', 'hint': 'np.round(np.linalg.det(M), 6). Check it by cofactor expansion along row 1.', 'ok': 'Because det(M) ≠ 0, M has an inverse.'},
    '6.5': {'h': '68f830fb53ffe49d', 'hint': 'det [[a, b], [c, d]] = ad − bc. Set 1 × k − 2 × 3 = 0.', 'ok': 'When k has this value, row 2 is a multiple of row 1.'},
    '6.6': {'h': '7f0c01fb9256da13', 'hint': 'Each row of A3 holds the coefficients of x, y and z in one equation. Remember −y has coefficient −1.', 'ok': 'Substitute x, y and z back into the equations to check.'},
    '6.7': {'h': '3f191bb4534e8522', 'hint': 'First check M.shape[0] == M.shape[1]; a non-square matrix cannot be symmetric.', 'args': [([[1, 2], [2, 1]],), ([[1, 2], [3, 1]],), ([[1, 2, 3]],), ([[5]],), ([[1, 0, 2], [0, 3, 4], [2, 4, 5]],)]},
    '6.8': {'h': '6c95ef22eb4be7e4', 'hint': 'The matrix must be square, and np.tril(M, -1) must contain only zeros: use not np.tril(M, -1).any().', 'ok': 'Diagonal matrices are both upper and lower triangular.', 'args': [([[1, 2], [0, 3]],), ([[1, 0], [2, 3]],), ([[1, 2, 3], [0, 4, 5], [0, 0, 6]],), ([[1, 2, 3], [0, 4, 5]],), ([[7]],)]},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
**NumPy** is the standard free Python library for matrices. A matrix is written as a list of rows. `A.shape` gives its order (rows, columns).

In [ ]:
import numpy as np
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[0, 1],
              [1, 0]])
print('A =\n', A, '\norder:', A.shape)
print('A + B =\n', A + B)
print('A @ B (matrix product) =\n', A @ B)
print('B @ A =\n', B @ A)

**Watch out:** in NumPy, `A * B` multiplies entry by entry, which is **not** the matrix product. Always use `@` for the matrix product. Notice also that **AB ≠ BA**: matrix multiplication is not commutative.

`np.linalg.det` gives the determinant, and `np.linalg.inv` gives the inverse. Computers work with decimals, so tiny rounding errors such as `-2.0000000000000004` are normal. `np.round(x, 6)` tidies them up.

In [ ]:
print('det(A) =', np.round(np.linalg.det(A), 6))
Ainv = np.linalg.inv(A)
print('A⁻¹ =\n', Ainv)
print('A @ A⁻¹ =\n', np.round(A @ Ainv, 6))     # should be the identity matrix I

## Part B – Practise (about 20 minutes)
Use these matrices (run the cell first):

In [ ]:
P = np.array([[3, 1, 2],
              [1, 0, 1]])
Q = np.array([[1, 1],
              [2, 1],
              [3, 1]])
M = np.array([[2, 0, 1],
              [1, 3, 2],
              [1, 1, 1]])

In [ ]:
answer_6_1 = ...   # the order (shape) of P, as a tuple

In [ ]:
check('6.1', answer_6_1)

In [ ]:
answer_6_2 = ...   # the matrix product PQ

In [ ]:
check('6.2', answer_6_2)

In [ ]:
answer_6_3 = ...   # the transpose of P (use P.T)

In [ ]:
check('6.3', answer_6_3)

In [ ]:
answer_6_4 = ...   # the determinant of M, rounded with np.round(..., 6)

In [ ]:
check('6.4', answer_6_4)

**B5. When is a matrix singular?** A matrix with determinant 0 is **singular** and has no inverse. For which value of k is [[1, 2], [3, k]] singular? Try values in a loop, or solve 1·k − 2·3 = 0.

In [ ]:
answer_6_5 = ...

In [ ]:
check('6.5', answer_6_5)

## Part C – Build a tool (about 20 minutes)
### C1. Solving a system of equations
The system
- x + y + z = 6
- 2x − y + z = 3
- x + 2y − z = 2

can be written as **AX = B**, so X = A⁻¹B. Set up A and B, then solve with `np.linalg.solve(A, B)` (which is faster and more accurate than computing the inverse). Round the answer with `np.round(..., 6)`.

In [ ]:
A3 = np.array(...)     # YOUR CODE: the coefficient matrix
B3 = np.array(...)     # YOUR CODE: the right-hand sides
answer_6_6 = np.round(np.linalg.solve(A3, B3), 6) if A3.shape != () else ...
print(answer_6_6)

In [ ]:
check('6.6', answer_6_6)

### C2. Special matrices
Complete the two tests from Study Session 6.3. Each takes a NumPy array and returns `True` or `False`.
- A matrix is **symmetric** if it is square and equal to its transpose: `np.array_equal(M, M.T)`.
- A matrix is **upper triangular** if it is square and every entry **below** the main diagonal is 0. `np.tril(M, -1)` gives just the entries below the diagonal.

In [ ]:
def is_symmetric(M):
    M = np.array(M)
    ...      # YOUR CODE (remember: it must be square)

In [ ]:
check('6.7', is_symmetric)

In [ ]:
def is_upper_triangular(M):
    M = np.array(M)
    ...      # YOUR CODE

In [ ]:
check('6.8', is_upper_triangular)

## Part D – Apply and reflect (about 10 minutes)
**D1. Prepare for Assignment 6.** Write your assignment system in matrix form AX = B in the cell. Solve it **by hand first**, then use NumPy to check your determinant, your inverse and your solution.

In [ ]:
# D1: check your Assignment 6 working

**D2. Reflect.** Answer in one or two sentences each:
1. Give an example from Part A or B that shows AB ≠ BA.
2. Why does a matrix with determinant 0 have no inverse? What does that mean for the system of equations?
3. Why does NumPy sometimes print 1.9999999999999998 instead of 2?

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `csc203-labs` repository → commit message *"Lab 6 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.